# Silver — Data Quality Checks
Runs after all silver tables are written and **before** gold. Every check is a rule the data must satisfy; the number of violating rows is recorded in `silver.dq_results`.

- `error` checks stop the pipeline when they fail (gold is not rebuilt on bad data)
- `warn` checks are recorded only

## Init

In [ ]:
import os, sys

root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.dq import Check, WARN, not_null, unique, references, accepted_values, run_checks, failed_errors

CATALOG = "workspace"
S = f"{CATALOG}.silver"

## Define checks
Grouped by what they protect: keys, joins, and business rules.

In [ ]:
CHECKS = [
    # --- primary keys: gold surrogate keys are built on these ---
    not_null(f"{S}.crm_customers", "customer_id"),
    unique(f"{S}.crm_customers", "customer_id"),
    not_null(f"{S}.crm_products", "product_number"),
    not_null(f"{S}.crm_sales", "order_number"),
    unique(f"{S}.erp_product_category", "category_id"),

    # --- foreign keys: every fact row must find its dimension ---
    references(f"{S}.crm_sales", "customer_id", f"{S}.crm_customers", "customer_id"),
    references(f"{S}.crm_sales", "product_number", f"{S}.crm_products", "product_number"),
    references(f"{S}.crm_products", "category_id", f"{S}.erp_product_category", "category_id", severity=WARN),

    # --- business rules ---
    accepted_values(f"{S}.crm_customers", "gender", ["Male", "Female", "n/a"]),
    accepted_values(f"{S}.crm_customers", "marital_status", ["Single", "Married", "n/a"]),
    Check("quantity_positive", f"{S}.crm_sales", "quantity > 0"),
    Check("sales_amount_positive", f"{S}.crm_sales", "sales_amount > 0"),
    Check("order_before_ship", f"{S}.crm_sales", "order_date IS NULL OR ship_date IS NULL OR order_date <= ship_date", severity=WARN),
    Check("birth_date_not_future", f"{S}.erp_customers", "birth_date IS NULL OR birth_date <= current_date()"),
]

## Run and persist results

In [ ]:
results = run_checks(spark, CHECKS)
results.display()

results.write.mode("append").format("delta").saveAsTable(f"{S}.dq_results")

## Gate
Fail the task (and therefore the job) if any `error` check has violations.

In [ ]:
failed = failed_errors(results)
if failed:
    raise Exception(f"Data quality checks failed: {failed}")
print(f"All {len(CHECKS)} checks passed (or warned)")

## History

In [ ]:
%sql
SELECT run_at, check_name, severity, violations, passed
FROM workspace.silver.dq_results
ORDER BY run_at DESC, check_name
LIMIT 50;